# دریافت گروهی و مدیریت داده

0.6.0a2. داده‌های نمونه مصنوعی هستند؛ قیمت واقعی بازار نیستند.

## 1 - ورود کتابخانه‌ها

In [ ]:
import pandas as pd
import pydara
from pydara import tse
from pathlib import Path
from tempfile import TemporaryDirectory
from pydara.tse.samples.extended import sample_extended_client
print(pydara.__version__)
assert pydara.__version__ == "0.7.3a6"

## 2 - تعریف توابع

In [ ]:
def show(frame):
    display(tse.select_columns(frame).head())
    return len(frame)

## 3 - تنظیمات

In [ ]:
SYMBOL = 'فولاد'
START = '1403-12-27'
END = '1403-12-28'

## 4 - اجرا

In [ ]:
with TemporaryDirectory(prefix='pydara-demo-') as tmp:
    path = Path(tmp)
    with sample_extended_client(storage=True, data_dir=path/'db') as client:
        job = client.download([SYMBOL, SYMBOL], start=START, end=END, update=False, checkpoint_dir=path/'jobs')
        resumed = client.download([SYMBOL, SYMBOL], start=START, end=END, update=False, checkpoint_dir=path/'jobs')
        display(job.requests)
        display(resumed.requests)
        assert resumed.requests['resumed'].all()
        files = tse.export_many(job, path/'exports', formats=['csv', 'sqlite'])
        display(files)
        client.flush(timeout=10)
        display(client.datasets())
        print(client.backup(path/'backup.sqlite'))

## 5 - گزارش نتیجه

In [ ]:
display(tse.help('download'))
display(tse.coverage(tse.load_sample()))
print('Offline workflow OK; no real server used.')